
# MCM Training — Google Colab

> ⚠️ **This notebook must be opened and run in Google Colab in your browser.**
> Do NOT run it from VS Code — it requires a GPU, `google.colab`, and the Colab Python environment.
>
> **To open in Colab:**
> 1. Go to [colab.research.google.com](https://colab.research.google.com)
> 2. File → Open notebook → Upload → select this `.ipynb` file
> 3. Runtime → Change runtime type → **T4 GPU**
> 4. Then run cells top to bottom

## Steps
1. Mount Google Drive
2. Upload your `data/synthetic/` folder to Drive at `My Drive/mcm/data/synthetic/` (do this once from your laptop)
3. Run cells top to bottom
4. Checkpoints save to Drive — safe to disconnect and resume later

### Upload data to Drive (from your terminal before opening Colab):
```bash
# Drag-and-drop data/synthetic/ into My Drive/mcm/data/synthetic/ via the Drive web UI
# Files needed: train.jsonl, val.jsonl, test.jsonl
```


## 1. Verify GPU

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

Sat Apr 18 07:29:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [2]:
from google.colab import drive
# drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/mcm/data/synthetic'
OUTPUT_DIR = '/content/drive/MyDrive/mcm/checkpoints/mcm-write-v1'

# Verify data is present
for fname in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    path = os.path.join(DATA_DIR, fname)
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else -1
    status = f'{size_mb:.1f} MB' if size_mb > 0 else '❌ MISSING'
    print(f'{fname}: {status}')

train.jsonl: 118.3 MB
val.jsonl: 14.7 MB
test.jsonl: 14.3 MB


## 3. Install dependencies

This installs into the Colab runtime (not Drive). Must re-run if runtime resets.

In [ ]:

# Install/upgrade only the ML packages we need.
# Do NOT touch numpy — Colab's pre-installed numpy is fine and touching it
# causes binary incompatibility errors with other Colab packages (numba, opencv, etc).
!pip install -q --upgrade \
    'transformers>=4.43.0' \
    'peft>=0.11.0' \
    'trl>=0.9.0' \
    'bitsandbytes>=0.43.0' \
    'datasets>=2.20.0' \
    'accelerate>=0.31.0' \
    'sentencepiece>=0.2.0'

import trl, peft, transformers, numpy as np
print(f'trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}  numpy={np.__version__}')
print('Versions printed above. If imports fail, use Runtime → Restart session, then re-run from cell 1.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.7 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 103.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 130.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.4 which i

/usr/local/lib/python3.12/dist-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: module 'numpy._globals' has no attribute '_signature_descriptor' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

## 4. (Optional) W&B login for training metrics

Skip this cell if you don't want W&B tracking — training will still work.

In [ ]:
import os
# Option A: paste your W&B API key here
# os.environ['WANDB_API_KEY'] = 'your-wandb-key-here'

# Option B: disable W&B entirely
os.environ['WANDB_DISABLED'] = 'true'

print('W&B tracking disabled — remove the line above and set WANDB_API_KEY to enable.')

## 5. Clone repo (or copy training script directly)

Option A: clone from GitHub (if repo is public or you add credentials).
Option B: upload `scripts/train_mcm.py` manually to `/content/` via the Colab File panel.

In [7]:
import os

# ---------- Option A: clone from GitHub ----------
# !git clone https://github.com/YOUR_USERNAME/learned-compiler-memory /content/repo
# SCRIPT = '/content/repo/scripts/train_mcm.py'

# ---------- Option B: copy from Drive (if you put the repo there) ----------
# !cp -r /content/drive/MyDrive/mcm/repo /content/repo
# SCRIPT = '/content/repo/scripts/train_mcm.py'

# ---------- Option C: write the script inline (auto-generated below) ----------
SCRIPT = '/content/train_mcm.py'

print(f'Will use script: {SCRIPT}')

Will use script: /content/train_mcm.py


In [ ]:

# Write train_mcm.py — uses runtime API introspection to work with any TRL version
script_content = r'''
import argparse, inspect
from pathlib import Path
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

# --- Detect TRL API at runtime so the script works across all TRL versions ---
try:
    from trl import SFTConfig
    _sft_cfg_params = set(inspect.signature(SFTConfig.__init__).parameters)
    USE_SFT_CONFIG = True
    # TRL <= 0.15: max_seq_length; TRL >= 0.16: renamed to max_length
    MAX_SEQ_KWARG = "max_seq_length" if "max_seq_length" in _sft_cfg_params else "max_length"
    print(f"[API] SFTConfig detected, seq_len kwarg='{MAX_SEQ_KWARG}'")
except ImportError:
    USE_SFT_CONFIG = False
    print("[API] SFTConfig not found, using TrainingArguments (TRL <= 0.9)")

_trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
USE_PROCESSING_CLASS = "processing_class" in _trainer_params
print(f"[API] SFTTrainer tokenizer arg: {'processing_class' if USE_PROCESSING_CLASS else 'tokenizer'}")
# ---


def load_model_and_tokenizer(model_name, use_fp16):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16 if use_fp16 else torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    # use_gradient_checkpointing reduces activation memory at the cost of ~20% slower step
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    return model, tokenizer


def build_peft_model(model):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    return get_peft_model(model, lora_config)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--head", default="write", choices=["write", "read"])
    parser.add_argument("--base_model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--data_dir", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--per_device_batch_size", type=int, default=2)
    parser.add_argument("--grad_accumulation_steps", type=int, default=16)
    parser.add_argument("--max_seq_length", type=int, default=1024)
    parser.add_argument("--resume_from_checkpoint", type=str, default=None)
    args = parser.parse_args()

    if args.head == "read":
        args.max_seq_length = min(args.max_seq_length, 1024)

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16
    print(f"GPU dtype: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'cpu'}")

    data_dir = Path(args.data_dir)
    dataset = load_dataset(
        "json",
        data_files={
            "train": str(data_dir / "train.jsonl"),
            "validation": str(data_dir / "val.jsonl"),
        },
    )

    print(f"Loading {args.base_model}...")
    model, tokenizer = load_model_and_tokenizer(args.base_model, use_fp16)
    model = build_peft_model(model)
    model.print_trainable_parameters()

    def formatting_func(examples):
        return [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in examples["messages"]
        ]

    common_train_kwargs = dict(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.per_device_batch_size,
        gradient_accumulation_steps=args.grad_accumulation_steps,
        gradient_checkpointing=True,          # saves ~40% activation memory, ~20% slower
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        bf16=use_bf16,
        fp16=use_fp16,
        logging_steps=10,
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=True,
        report_to="none",
    )

    if USE_SFT_CONFIG:
        # Pre-map the chat template so SFTConfig can use dataset_text_field
        def apply_template(examples):
            return {"text": [
                tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                for msgs in examples["messages"]
            ]}
        dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])

        cfg_kwargs = {
            **common_train_kwargs,
            "dataset_text_field": "text",
            "eval_strategy": "steps",
            MAX_SEQ_KWARG: args.max_seq_length,
        }
        training_args = SFTConfig(**cfg_kwargs)
        trainer_kwargs = dict(
            model=model,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],
            args=training_args,
        )
    else:
        training_args = TrainingArguments(
            **common_train_kwargs,
            evaluation_strategy="steps",
        )
        trainer_kwargs = dict(
            model=model,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],
            args=training_args,
            formatting_func=formatting_func,
            max_seq_length=args.max_seq_length,
        )

    # tokenizer vs processing_class depending on TRL version
    tok_key = "processing_class" if USE_PROCESSING_CLASS else "tokenizer"
    trainer_kwargs[tok_key] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)

    print("Starting training...")
    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)
    trainer.save_model(args.output_dir)
    print(f"Model saved to {args.output_dir}")


if __name__ == "__main__":
    main()
'''

with open('/content/train_mcm.py', 'w') as f:
    f.write(script_content.strip())

print('train_mcm.py written to /content/')


train_mcm.py written to /content/



## 6. Train (Colab fallback)

> **Preferred path:** SSH to Lambda Labs or RunPod — no browser-babysitting, 10–20× faster GPU, ~$2–6 total.
> See `PLAN.md` section 2e and `scripts/setup_gpu_instance.sh` for the full walkthrough.

This cell is the Colab fallback. Checkpoints save every 200 steps (~1 hr on T4) to Drive.
**If the session disconnects:** set `RESUME_FROM_CHECKPOINT` to the latest checkpoint path and re-run this cell only.


In [17]:

import os

# Tell PyTorch to use expandable memory segments — reduces fragmentation on T4
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

DATA_DIR = '/content/drive/MyDrive/mcm/data/synthetic'
OUTPUT_DIR = '/content/drive/MyDrive/mcm/checkpoints/mcm-write-v1'
RESUME_FROM_CHECKPOINT = None  # e.g. OUTPUT_DIR + '/checkpoint-400'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# T4-safe settings:
#   max_seq_length 1024  — biggest lever: logits are vocab_size × seq × batch × 4B
#   per_device_batch_size 2 + grad_accumulation_steps 16 → effective batch = 32 (same as before)
# Use ! shell magic — streams stdout+stderr live directly into the cell output
if RESUME_FROM_CHECKPOINT:
    !PYTORCH_ALLOC_CONF=expandable_segments:True python /content/train_mcm.py \
        --head write \
        --data_dir {DATA_DIR} \
        --output_dir {OUTPUT_DIR} \
        --epochs 3 \
        --per_device_batch_size 2 \
        --grad_accumulation_steps 16 \
        --max_seq_length 1024 \
        --resume_from_checkpoint {RESUME_FROM_CHECKPOINT}
else:
    !PYTORCH_ALLOC_CONF=expandable_segments:True python /content/train_mcm.py \
        --head write \
        --data_dir {DATA_DIR} \
        --output_dir {OUTPUT_DIR} \
        --epochs 3 \
        --per_device_batch_size 2 \
        --grad_accumulation_steps 16 \
        --max_seq_length 1024


2026-04-18 07:19:15.042720: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776496755.064495    7172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776496755.070707    7172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776496755.086021    7172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776496755.086064    7172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776496755.086068    7172 computation_placer.cc:177] computation placer alr

## 7. Verify the saved adapter

In [ ]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/mcm/checkpoints/mcm-write-v1'
files = os.listdir(OUTPUT_DIR)
print('Files in output dir:')
for f in sorted(files):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

## 8. Quick inference test

Smoke test: load the saved adapter and run one inference.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER_PATH = '/content/drive/MyDrive/mcm/checkpoints/mcm-write-v1'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

TEST_LOG = """[Turn 1] User: Remind me to buy milk tomorrow.
[Turn 2] Assistant: Sure, I'll remind you about milk tomorrow morning.
[Turn 3] User: Also, I have a dentist appointment at 3pm.
[Turn 4] Assistant: Got it — dentist at 3pm."""

messages = [
    {"role": "system", "content": "You are a memory consolidation model. Convert raw agent interaction logs into structured memory JSON."},
    {"role": "user", "content": TEST_LOG},
]

input_ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors='pt'
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=512,
        do_sample=False,
        temperature=1.0,
    )

response = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
print('MCM output:')
print(response)

## 9. (Optional) Push adapter to HuggingFace Hub

Push the LoRA adapter to HF Hub so you can load it from anywhere for experiments — no Colab session needed.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # will prompt for HF token

In [ ]:
HF_REPO = 'YOUR_HF_USERNAME/mcm-write-v1'  # change this

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f'Adapter pushed to https://huggingface.co/{HF_REPO}')
print('Load anywhere with: PeftModel.from_pretrained(base_model, "' + HF_REPO + '")')